# Módulo 3 — Visualização de Indicadores

Dados bem visualizados comunicam insights que tabelas não conseguem. Neste notebook criamos gráficos profissionais para indicadores do setor elétrico.

## O que vamos criar

1. **matplotlib** — Gráfico de barras: consumidores por classe tarifária
2. **seaborn** — Heatmap: qualidade de dados por campo e município
3. **plotly** — Gráfico interativo: tendência de consumo mensal
4. **plotly** — Indicadores DEC/FEC vs. limites ANEEL

---

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import seaborn as sns
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots

# Configurações globais
plt.rcParams.update({
    "figure.facecolor": "white",
    "axes.facecolor": "#f8f9fa",
    "axes.grid": True,
    "grid.alpha": 0.4,
    "font.family": "sans-serif",
})
sns.set_theme(style="whitegrid", palette="muted")

# Carregar dados
df = pd.read_csv("../modulo2_dados_com_pandas/dados/consumidores_simulado.csv")
print(f"Dados carregados: {len(df)} registros")

## 1. matplotlib — Gráfico de Barras por Classe

In [ ]:
# Preparar dados
por_classe = (
    df.groupby("cod_classe_consumo")["cod_consumidor"]
    .count()
    .sort_values(ascending=False)
)

# Paleta de cores para o setor elétrico
cores = {
    "RESIDENCIAL": "#3498db",
    "COMERCIAL": "#2ecc71",
    "INDUSTRIAL": "#e74c3c",
    "RURAL": "#f39c12",
    "PODER PUBLICO": "#9b59b6",
    "ILUMINACAO PUBLICA": "#1abc9c",
    "SERVICO PUBLICO": "#34495e",
}

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# --- Gráfico 1: Barras verticais — contagem ---
ax1 = axes[0]
cores_barras = [cores.get(c, "#95a5a6") for c in por_classe.index]
bars = ax1.bar(por_classe.index, por_classe.values, color=cores_barras, edgecolor="white", linewidth=0.5)

# Adicionar valores nas barras
for bar, val in zip(bars, por_classe.values):
    ax1.text(
        bar.get_x() + bar.get_width() / 2,
        bar.get_height() + 0.1,
        str(val),
        ha="center", va="bottom", fontsize=9, fontweight="bold"
    )

ax1.set_title("Consumidores por Classe de Consumo", fontsize=13, fontweight="bold", pad=12)
ax1.set_xlabel("Classe")
ax1.set_ylabel("Quantidade de UCs")
ax1.tick_params(axis="x", rotation=30)
ax1.set_ylim(0, por_classe.max() * 1.15)

# --- Gráfico 2: Pizza --- 
ax2 = axes[1]
wedges, texts, autotexts = ax2.pie(
    por_classe.values,
    labels=por_classe.index,
    autopct="%1.1f%%",
    colors=[cores.get(c, "#95a5a6") for c in por_classe.index],
    startangle=90,
    pctdistance=0.8
)
for text in autotexts:
    text.set_fontsize(8)
ax2.set_title("Participação por Classe (%)", fontsize=13, fontweight="bold", pad=12)

plt.tight_layout()
plt.savefig("grafico_classes.png", dpi=150, bbox_inches="tight")
plt.show()
print("Gráfico salvo: grafico_classes.png")

## 2. seaborn — Heatmap de Qualidade de Dados

In [ ]:
# Calcular % de nulos por coluna e município
colunas_analise = ["cpf_cnpj", "num_medidor", "dat_ultima_leitura", "num_cep", "vlr_consumo_medio_kwh"]

# Pegar municípios com mais registros
top_municipios = df["nom_municipio"].value_counts().head(8).index.tolist()
df_top = df[df["nom_municipio"].isin(top_municipios)]

# Calcular % de nulos por município e coluna
heatmap_data = pd.DataFrame()
for col in colunas_analise:
    pct_nulos = df_top.groupby("nom_municipio")[col].apply(
        lambda x: x.isnull().sum() / len(x) * 100
    ).round(1)
    heatmap_data[col] = pct_nulos

# Renomear colunas para exibição
heatmap_data.columns = ["CPF/CNPJ", "Medidor", "Dt. Leitura", "CEP", "Consumo"]

fig, ax = plt.subplots(figsize=(10, 6))

sns.heatmap(
    heatmap_data,
    annot=True,
    fmt=".0f",
    cmap="RdYlGn_r",  # verde=bom, vermelho=problema
    vmin=0, vmax=100,
    linewidths=0.5,
    linecolor="white",
    ax=ax,
    annot_kws={"size": 10}
)

ax.set_title(
    "Percentual de Campos Nulos por Município (%)\nVermelho = maior incidência de nulos",
    fontsize=12, fontweight="bold", pad=14
)
ax.set_xlabel("Campo do Cadastro", fontsize=11)
ax.set_ylabel("Município", fontsize=11)
ax.tick_params(axis="x", rotation=20)
ax.tick_params(axis="y", rotation=0)

# Colorbar label
cbar = ax.collections[0].colorbar
cbar.set_label("% Nulos", fontsize=10)

plt.tight_layout()
plt.savefig("heatmap_qualidade.png", dpi=150, bbox_inches="tight")
plt.show()
print("Heatmap salvo: heatmap_qualidade.png")

## 3. plotly — Tendência de Consumo Mensal (Interativo)

In [ ]:
# Gerar dados simulados de consumo mensal por classe
np.random.seed(42)
meses = pd.date_range("2024-01", "2025-02", freq="MS")

consumo_base = {
    "RESIDENCIAL": 250,
    "COMERCIAL": 3000,
    "INDUSTRIAL": 45000,
    "RURAL": 400,
}

# Criar DataFrame de série temporal simulada
registros = []
for mes in meses:
    for classe, base in consumo_base.items():
        # Sazonalidade: maior consumo no verão (dez-mar) para residencial
        sazonalidade = 1.0
        if classe == "RESIDENCIAL" and mes.month in [12, 1, 2, 3]:
            sazonalidade = 1.25
        elif classe == "INDUSTRIAL" and mes.month in [7, 8]:
            sazonalidade = 0.85
        
        consumo = base * sazonalidade * (1 + np.random.normal(0, 0.05))
        registros.append({
            "mes": mes,
            "classe": classe,
            "consumo_medio_kwh": round(consumo, 1)
        })

df_temporal = pd.DataFrame(registros)

# Gráfico interativo com plotly
fig = px.line(
    df_temporal,
    x="mes",
    y="consumo_medio_kwh",
    color="classe",
    title="Consumo Médio Mensal por Classe (kWh)",
    labels={
        "mes": "Mês",
        "consumo_medio_kwh": "Consumo Médio (kWh)",
        "classe": "Classe"
    },
    template="plotly_white",
    markers=True,
    color_discrete_map={
        "RESIDENCIAL": "#3498db",
        "COMERCIAL": "#2ecc71",
        "INDUSTRIAL": "#e74c3c",
        "RURAL": "#f39c12",
    }
)

fig.update_layout(
    xaxis_title="Mês",
    yaxis_title="Consumo Médio (kWh)",
    legend_title="Classe",
    hovermode="x unified",
    height=450
)

fig.update_xaxes(dtick="M1", tickformat="%b/%Y", tickangle=30)
fig.show()

## 4. plotly — Indicadores DEC/FEC vs. Limites ANEEL

In [ ]:
# Dados simulados de DEC/FEC mensal
np.random.seed(123)
meses_label = ["Jan", "Fev", "Mar", "Abr", "Mai", "Jun",
                "Jul", "Ago", "Set", "Out", "Nov", "Dez"]

dec_real = [1.2, 0.8, 2.1, 1.5, 0.9, 3.2, 1.8, 1.1, 0.7, 1.4, 1.9, 2.3]
fec_real = [1.1, 0.9, 1.8, 1.3, 0.8, 2.5, 1.6, 1.0, 0.9, 1.2, 1.5, 1.7]

# Limites fictícios para exemplo (limites reais variam por conjunto/distribuidora)
limite_dec_mensal = 1.8
limite_fec_mensal = 1.5

# DEC acumulado
dec_acumulado = [sum(dec_real[:i+1]) for i in range(12)]
limite_dec_anual = 12.0

# Criar subplots
fig = make_subplots(
    rows=2, cols=2,
    subplot_titles=(
        "DEC Mensal vs. Limite",
        "FEC Mensal vs. Limite",
        "DEC Acumulado no Ano",
        "DEC vs. FEC (dispersão)"
    ),
    vertical_spacing=0.14,
    horizontal_spacing=0.1
)

# --- Plot 1: DEC Mensal ---
cores_dec = ["#e74c3c" if v > limite_dec_mensal else "#3498db" for v in dec_real]
fig.add_trace(go.Bar(
    x=meses_label, y=dec_real,
    name="DEC Real",
    marker_color=cores_dec,
    showlegend=False
), row=1, col=1)
fig.add_trace(go.Scatter(
    x=meses_label, y=[limite_dec_mensal]*12,
    name="Limite DEC", mode="lines",
    line=dict(color="#e74c3c", dash="dash", width=2),
), row=1, col=1)

# --- Plot 2: FEC Mensal ---
cores_fec = ["#e74c3c" if v > limite_fec_mensal else "#2ecc71" for v in fec_real]
fig.add_trace(go.Bar(
    x=meses_label, y=fec_real,
    name="FEC Real",
    marker_color=cores_fec,
    showlegend=False
), row=1, col=2)
fig.add_trace(go.Scatter(
    x=meses_label, y=[limite_fec_mensal]*12,
    name="Limite FEC", mode="lines",
    line=dict(color="#e74c3c", dash="dash", width=2),
), row=1, col=2)

# --- Plot 3: DEC Acumulado ---
fig.add_trace(go.Scatter(
    x=meses_label, y=dec_acumulado,
    name="DEC Acumulado", mode="lines+markers",
    line=dict(color="#3498db", width=2),
    fill="tozeroy", fillcolor="rgba(52,152,219,0.2)",
    showlegend=False
), row=2, col=1)
fig.add_trace(go.Scatter(
    x=meses_label, y=[limite_dec_anual]*12,
    name="Limite Anual", mode="lines",
    line=dict(color="#e74c3c", dash="dot", width=2),
    showlegend=False
), row=2, col=1)

# --- Plot 4: Dispersão DEC x FEC ---
fig.add_trace(go.Scatter(
    x=dec_real, y=fec_real,
    mode="markers+text",
    text=meses_label,
    textposition="top center",
    marker=dict(size=10, color="#9b59b6"),
    name="DEC vs FEC",
    showlegend=False
), row=2, col=2)

fig.update_layout(
    title_text="Painel de Indicadores DEC/FEC — Conjunto de Distribuição XYZ",
    title_font_size=15,
    height=600,
    template="plotly_white",
    legend=dict(orientation="h", yanchor="bottom", y=1.02, xanchor="right", x=1)
)

# Atualizar eixos
fig.update_yaxes(title_text="Horas", row=1, col=1)
fig.update_yaxes(title_text="Interrupções", row=1, col=2)
fig.update_yaxes(title_text="Horas acumuladas", row=2, col=1)
fig.update_xaxes(title_text="DEC (h)", row=2, col=2)
fig.update_yaxes(title_text="FEC (interr.)", row=2, col=2)

fig.show()

## 5. Resumo: Qual biblioteca usar?

In [ ]:
resumo = {
    "Biblioteca": ["matplotlib", "seaborn", "plotly"],
    "Melhor para": [
        "Relatórios em PDF/Word, exportação de imagens de alta resolução",
        "Heatmaps de correlação/qualidade, análises estatísticas",
        "Dashboards interativos, apresentações, análise exploratória"
    ],
    "Interativo": ["Não", "Não", "Sim"],
    "Curva de aprendizado": ["Média", "Baixa", "Baixa"],
}

df_resumo = pd.DataFrame(resumo)
print("Comparação das bibliotecas de visualização:")
df_resumo